# predicting customer churn

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import  ColumnTransformer
from sklearn.metrics import r2_score, log_loss

from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier

from tqdm import tqdm  # Provides the progress of model running

import os
os.chdir("C:/Users/PGCP-AI/Downloads/playground-series-s6e3") #fetch the dataset from the folder

In [2]:
train = pd.read_csv("train.csv", index_col=0)
train.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
id,,,,,,,,,,,,,,,,,,,,
0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,No,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,Yes,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [4]:
le = LabelEncoder()
train['Churn'] = le.fit_transform(train['Churn'])

X = train.drop('Churn',axis=1)
y=train['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=26, 
                                                    stratify=train['Churn'])

In [5]:
from sklearn.compose import make_column_selector

ohe=OneHotEncoder(sparse_output=False,drop="first").set_output(transform="pandas")

trans = ColumnTransformer(
    transformers=[("OHE", ohe, make_column_selector(dtype_include=object))], remainder="passthrough",
    verbose_feature_names_out=False).set_output(transform="pandas")

X_trn_ohe = trans.fit_transform(X_train)
X_tst_ohe = trans.transform(X_test)

In [6]:
# Ks=[1,2,3,4,5,6,7,8,9,10]
# scores=[]
# for k in Ks:
#     knn = KNeighborsRegressor(n_neighbors=k)
#     knn.fit(X_trn_ohe, y_train)
#     y_pred = knn.predict(X_tst_ohe)
#     scores.append([k, r2_score(y_test, y_pred)])
# df_scores = pd.DataFrame(scores, columns=['Ks', 'score'])
# df_scores.sort_values('score', ascending=False)

In [7]:
# Standard Scaler
scaler = StandardScaler().set_output(transform="pandas")

scaler.fit(X_trn_ohe)   # Means and SD get calculated
X_trn_scl = scaler.transform(X_trn_ohe)  # Actual transformation of data
X_tst_scl = scaler.transform(X_tst_ohe)

Ks=[1,2,3,4,5,6,7,8,9,10]
scores=[]

for k in tqdm(Ks):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_trn_scl, y_train)
    y_pred_prob = knn.predict_proba(X_tst_scl)
    scores.append([k, log_loss(y_test, y_pred_prob)])
    
df_scores = pd.DataFrame(scores, columns=['Ks', 'Log_loss score'])
df_scores.sort_values('Log_loss score', ascending=True)

100%|█████████████████████████████████████████████████████████████████████████████████| 10/10 [22:30<00:00, 135.08s/it]


,Ks,Log_loss score
9,10,0.726170
8,9,0.786718
7,8,0.866691
6,7,0.974218
5,6,1.130664
4,5,1.361721
3,4,1.718048
2,3,2.336121
1,2,3.580989
0,1,6.921245


In [8]:
# MinMax Scaler
# scaler = MinMaxScaler().set_output(transform="pandas")

# scaler.fit(X_train)   # Means and SD get calculated
# X_trn_scl = scaler.transform(X_train)  # Actual transformation of data
# X_tst_scl = scaler.transform(X_test)

# Ks=[1,2,3,4,5,6,7,8,9,10]
# scores=[]
# for k in Ks:
#     knn = KNeighborsRegressor(n_neighbors=k)
#     knn.fit(X_trn_scl, y_train)
#     y_pred = knn.predict(X_tst_scl)
#     scores.append([k, r2_score(y_test, y_pred)])
# df_scores = pd.DataFrame(scores, columns=['Ks', 'R2 score'])
# df_scores.sort_values('R2 score', ascending=False)

## Inferencing

In [9]:
X_ohe = trans.fit_transform(X)
X_scl = scaler.fit_transform(X_ohe)
bm = KNeighborsClassifier(n_neighbors = 10, n_jobs = -1)
bm.fit(X_scl, y)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",10
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",-1


In [10]:
test = pd.read_csv("test.csv", index_col = 0)
test_ohe = trans.transform(test)
test_scl = scaler.transform(test_ohe)

In [11]:
submit = pd.read_csv("Sample_submission.csv")
y_pred_prob = bm.predict_proba(test_scl)
submit['Churn'] = y_pred_prob[:,1]

In [13]:
submit

,id,Churn
0,594194,0.1
1,594195,0.0
2,594196,0.1
3,594197,0.0
4,594198,0.5
...,...,...
254650,848844,0.0
254651,848845,0.7
254652,848846,0.4
254653,848847,0.0


In [12]:
submit.to_csv("sbt_knn_20_Apr.csv", index=False)

In [15]:
submit[submit['Churn'] >= 0.5]

,id,Churn
4,594198,0.5
6,594200,1.0
13,594207,0.9
16,594210,0.5
18,594212,0.8
...,...,...
254625,848819,0.9
254632,848826,0.8
254638,848832,1.0
254643,848837,0.7
